# Stable seat count and churn

**Question.** When a client has held the same seat count for at least 3 months, does that seat count separate clients who churn out within 9, 12, 15 or 18 months from clients who don't? If it does, is there a seat count that works as the dividing line?

**Source.** `SC Engagement List2026-11-15 09_48_04 (1).xlsx`, one row per staff engagement. This is a fresh analysis: none of the rules from `churn_analysis_raw.ipynb` are reused. Every assumption is listed in the last cell.

**Approach.**
1. Clean the engagement list and keep external client seats only.
2. Turn each engagement into a timeline of events: a seat starts, a seat ends.
3. Add the events up into a seat count per client per month.
4. Find each **stable spell**: the same non-zero seat count at 3 consecutive month-ends. Take the 3rd month as the anchor, the first point at which we know the count has held for 3 months.
5. From each anchor, check whether the client **churned out** (dropped to 0 seats and stayed there) within each horizon.
6. Test whether the anchor seat count separates churners from non-churners, and find the best dividing line.

**Part 3** adds the JP title / category of 1-seat clients' seats (from the Staff Central staff report) and follows clients that start with 1 seat to see who grows.

Outputs show counts only, never client names or shortcodes.

In [ ]:
# 1 · Setup
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

DATA = Path.home() / 'Desktop/Projects/Churn Analysis/Data/SC Engagement List2026-11-15 09_48_04 (1).xlsx'
EXTRACT = pd.Timestamp('2026-11-15')            # file timestamp
LAST_MONTH = pd.Period('2026-10', 'M')          # last complete month-end before the extract
HORIZONS = [9, 12, 15, 18]                      # months after the anchor
STABLE_MONTHS = 3                               # same seat count at this many consecutive month-ends
EXIT_CONFIRM_MONTHS = 3                         # a drop to 0 counts as churn only if it stays 0 this many month-ends
ANCHOR_FROM = pd.Period('2018-01', 'M')         # earliest anchor month analysed
BANDS = [(1, 1, '1'), (2, 2, '2'), (3, 4, '3–4'), (5, 9, '5–9'), (10, 19, '10–19'), (20, 10**6, '20+')]
H_COLOUR = {9: '#007AFF', 12: '#00AB9C', 15: '#F46600', 18: '#4121A9'}
SEED = 7

def band(n):
    for lo, hi, lbl in BANDS:
        if lo <= n <= hi:
            return lbl

def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n; c = p + z*z/(2*n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((c-h)/d, (c+h)/d)

In [ ]:
# 2 · Load and clean the engagement list
raw = pd.read_excel(DATA, dtype=str)
qa = []
def step(label, n):
    qa.append({'step': label, 'rows': int(n)})

step('Source rows', len(raw))
df = raw.copy()
df['start'] = pd.to_datetime(df['Engagement Date'], errors='coerce')
df['end'] = pd.to_datetime(df['Disengagement Date'], errors='coerce')

drop = df['Engagement ID'].duplicated(keep='first')
step('Dropped: duplicate Engagement ID (first kept)', drop.sum()); df = df[~drop]

# Internal and test accounts: every shortcode starting "CS" (CS Marketing, Disengaged Staff Pool = CSTREENGAGE,
# CS Now, test accounts CSS...), plus Cloudstaff-named accounts under other codes (CLOUDSTAFF, CIB107 Cloudstaff Hr, ...).
INTERNAL_NAME = re.compile(r'cloudstaff|staff pool|internal business|pd training', re.I)
internal = df['Shortcode'].fillna('').str.upper().str.startswith('CS') | df['Client Name'].fillna('').str.contains(INTERNAL_NAME)
step('Dropped: internal or test accounts', internal.sum()); df = df[~internal]

drop = df['Shortcode'].isna()
step('Dropped: no shortcode', drop.sum()); df = df[~drop]

drop = df['Engagement Status'].eq('bench')
step('Dropped: engagement status "bench" (not a working seat)', drop.sum()); df = df[~drop]

drop = df['start'].isna()
step('Dropped: no engagement date', drop.sum()); df = df[~drop]

drop = df['end'].notna() & (df['end'] < df['start'])
step('Dropped: disengagement before engagement', drop.sum()); df = df[~drop]

drop = df['Engagement Status'].eq('disengage') & df['end'].isna()
step('Dropped: status "disengage" but no disengagement date', drop.sum()); df = df[~drop]

step('Engagements used (external client seats)', len(df))
eng = df[['Engagement ID', 'Shortcode', 'start', 'end']].rename(columns={'Shortcode': 'client'}).reset_index(drop=True)
print(f"{eng['client'].nunique():,} external clients · engagements from {eng['start'].min():%b %Y} to {eng['start'].max():%b %Y}")
pd.DataFrame(qa)

In [ ]:
# 3 · Timeline of seat events
# A seat counts at a month-end if the engagement is active on that day.
#   seat start: +1 in the month of the engagement date
#   seat end:   -1 in the month of the disengagement date, or the next month when it falls on the last day of a month
#               (the person was still in the seat at that month-end)
starts = pd.DataFrame({'client': eng['client'], 'month': eng['start'].dt.to_period('M'), 'event': 'seat start', 'delta': 1})
ended = eng[eng['end'].notna()]
end_month = ended['end'].dt.to_period('M')
end_month = end_month.where(~ended['end'].dt.is_month_end, end_month + 1)
ends = pd.DataFrame({'client': ended['client'], 'month': end_month, 'event': 'seat end', 'delta': -1})
events = pd.concat([starts, ends], ignore_index=True).sort_values(['client', 'month']).reset_index(drop=True)
events = events[events['month'] <= LAST_MONTH]

print(f"{len(events):,} events: {(events.event == 'seat start').sum():,} seat starts, {(events.event == 'seat end').sum():,} seat ends")
events.groupby([events['month'].dt.year.rename('year'), 'event']).size().unstack().loc[2014:].T

In [ ]:
# 4 · Monthly seat count per client (sum of events), checked against a direct count of active engagements
net = events.groupby(['client', 'month'])['delta'].sum()
frames = []
for client, s in net.groupby(level=0):
    s = s.droplevel(0)
    idx = pd.period_range(s.index.min(), LAST_MONTH, freq='M')
    frames.append(pd.DataFrame({'client': client, 'month': idx, 'seats': s.reindex(idx, fill_value=0).cumsum().to_numpy()}))
panel = pd.concat(frames, ignore_index=True)

# check: seat count from events = engagements active on the month-end, for three test months
for m in [pd.Period('2019-06', 'M'), pd.Period('2022-12', 'M'), LAST_MONTH]:
    me = m.to_timestamp(how='end').normalize()
    direct = eng[(eng['start'] <= me) & (eng['end'].isna() | (eng['end'] >= me))].groupby('client').size()
    from_events = panel[panel['month'] == m].set_index('client')['seats']
    from_events = from_events[from_events > 0]
    assert direct.sort_index().equals(from_events.sort_index().astype(direct.dtype)), m
    print(f"{m}: {direct.sum():,} seats at {len(direct):,} clients — events and direct count agree")
assert (panel['seats'] >= 0).all()

In [ ]:
# 5 · Per client-month: run length at the current seat count, and confirmed exits
panel = panel.sort_values(['client', 'month']).reset_index(drop=True)
panel['t'] = panel['month'].map(lambda p: p.ordinal)
g = panel.groupby('client', sort=False)
panel['run'] = panel.groupby([panel['client'], (panel['seats'] != g['seats'].shift()).cumsum()]).cumcount() + 1

# Confirmed exit: seats go from >0 to 0 and stay 0 for EXIT_CONFIRM_MONTHS month-ends (or until the data ends).
last_t = LAST_MONTH.ordinal
drop_to_zero = (panel['seats'] == 0) & (g['seats'].shift().fillna(0) > 0)
zero_run_ahead = []
for client, s in panel.groupby('client', sort=False)['seats']:
    a = s.to_numpy()
    z = np.zeros(len(a), dtype=int)                       # consecutive zeros starting at each position
    for i in range(len(a) - 1, -1, -1):
        z[i] = (z[i + 1] + 1 if i + 1 < len(a) else 1) if a[i] == 0 else 0
    zero_run_ahead.append(z)
panel['zeros_ahead'] = np.concatenate(zero_run_ahead)
months_left = last_t - panel['t'] + 1
panel['exit'] = drop_to_zero & (panel['zeros_ahead'] >= np.minimum(EXIT_CONFIRM_MONTHS, months_left))
panel['dip'] = drop_to_zero & ~panel['exit']             # dropped to 0 but came back within the confirmation window

# months until the next confirmed exit (after t)
nxt = []
for client, s in panel.groupby('client', sort=False):
    tt, ex = s['t'].to_numpy(), s['exit'].to_numpy()
    out = np.full(len(tt), np.nan); upcoming = np.nan
    for i in range(len(tt) - 1, -1, -1):
        out[i] = upcoming - tt[i] if not np.isnan(upcoming) else np.nan
        if ex[i]:
            upcoming = tt[i]
    nxt.append(out)
panel['months_to_exit'] = np.concatenate(nxt)
for H in HORIZONS:
    panel[f'obs_{H}'] = panel['t'] + H <= last_t
    panel[f'churn_{H}'] = panel['months_to_exit'] <= H

print(f"{panel['exit'].sum():,} confirmed exits · {panel['dip'].sum():,} drops to 0 that came back within {EXIT_CONFIRM_MONTHS} months (not churn)")
print(f"{panel.loc[panel.seats > 0, 'client'].nunique():,} clients ever active · {panel.groupby('client')['exit'].any().sum():,} with at least one confirmed exit")

In [ ]:
# 6 · Stable spells: same non-zero seat count at STABLE_MONTHS consecutive month-ends; anchor = the month it reaches STABLE_MONTHS
spells = panel[(panel['seats'] > 0) & (panel['run'] == STABLE_MONTHS) & (panel['month'] >= ANCHOR_FROM)].copy()
spells['band'] = pd.Categorical(spells['seats'].map(band), [b[2] for b in BANDS], ordered=True)

rows = []
for H in HORIZONS:
    s = spells[spells[f'obs_{H}']]
    rows.append({'horizon (months)': H, 'anchor months': f"{s.month.min()} to {s.month.max()}", 'stable spells': len(s),
                 'clients': s.client.nunique(), 'churned out': int(s[f'churn_{H}'].sum()), 'churn rate': round(s[f'churn_{H}'].mean(), 3)})
print(f"{len(spells):,} stable spells from {spells.client.nunique():,} clients (anchors from {ANCHOR_FROM}). Spells per client, median {spells.groupby('client').size().median():.0f}.")
pd.DataFrame(rows)

## Does the seat count separate churners from non-churners?

For each horizon, using spells with the full horizon observed:
- **AUC**: how well a lower seat count ranks churners above non-churners. 0.5 = coin flip, about 0.7 = useful. The 95% range comes from resampling **clients**, because one client can have several spells.
- **Mann-Whitney p**: whether the seat-count distributions of churners and non-churners differ.
- **Best dividing line**: the rule "churns if seats ≤ k" with the highest Youden's J (sensitivity + specificity − 1).

In [ ]:
# 7 · Separation test per horizon
rng = np.random.default_rng(SEED)

def separation(sp, H, boot=400):
    s = sp[sp[f'obs_{H}']]
    y, x = s[f'churn_{H}'].astype(int).to_numpy(), s['seats'].to_numpy()
    auc = roc_auc_score(y, -x)
    groups = s.groupby('client').indices; keys = np.array(list(groups))
    aucs = []
    for _ in range(boot):
        idx = np.concatenate([groups[k] for k in rng.choice(keys, len(keys))])
        if 0 < y[idx].sum() < len(idx):
            aucs.append(roc_auc_score(y[idx], -x[idx]))
    p = mannwhitneyu(x[y == 1], x[y == 0], alternative='two-sided').pvalue
    best = None
    for k in sorted(set(x[x <= 50])):
        pred = x <= k
        sens = (pred & (y == 1)).sum() / max(y.sum(), 1); spec = (~pred & (y == 0)).sum() / max((y == 0).sum(), 1)
        if best is None or sens + spec - 1 > best[1]:
            best = (k, sens + spec - 1, sens, spec, y[pred].mean(), y[~pred].mean(), pred.mean())
    k, j, sens, spec, r_lo, r_hi, share = best
    return {'horizon': H, 'spells': len(s), 'churn rate': y.mean(),
            'median seats · churned': np.median(x[y == 1]), 'median seats · stayed': np.median(x[y == 0]),
            'AUC': auc, 'AUC 95% range': f"{np.percentile(aucs, 2.5):.2f}–{np.percentile(aucs, 97.5):.2f}", 'Mann-Whitney p': p,
            'best line: seats ≤': int(k), 'churn rate at or below': r_lo, 'churn rate above': r_hi,
            'times as likely': r_lo / r_hi if r_hi else np.nan, 'share of spells at or below': share,
            'sensitivity': sens, 'specificity': spec}

sep = pd.DataFrame([separation(spells, H) for H in HORIZONS])
sep.style.format({'churn rate': '{:.1%}', 'AUC': '{:.2f}', 'Mann-Whitney p': '{:.1e}', 'churn rate at or below': '{:.1%}', 'churn rate above': '{:.1%}',
                  'times as likely': '{:.1f}×', 'share of spells at or below': '{:.0%}', 'sensitivity': '{:.0%}', 'specificity': '{:.0%}',
                  'median seats · churned': '{:.0f}', 'median seats · stayed': '{:.0f}'}).hide(axis='index')

### Beyond the 1-seat line

Two follow-up checks, both with the 12-month horizon:
- **2+ seats only**: does seat count still separate churners once 1-seat clients are left out?
- **A client's first stable spell vs later spells**: the first spell usually comes soon after a client starts, so it may behave differently.

In [ ]:
# 7b · Separation among 2+ seat spells, and first vs later spells (12-month horizon)
H = 12
r2 = separation(spells[spells['seats'] >= 2], H)
print(f"2+ seats: {r2['spells']:,} spells, churn {r2['churn rate']:.1%}, AUC {r2['AUC']:.2f} ({r2['AUC 95% range']}); "
      f"best line ≤ {r2['best line: seats ≤']} seats: {r2['churn rate at or below']:.1%} vs {r2['churn rate above']:.1%} above")
sp = spells[spells[f'obs_{H}']].sort_values('month').copy()
sp['spell'] = np.where(sp.groupby('client').cumcount() == 0, 'first spell', 'later spell')
fl = sp.groupby(['band', 'spell'], observed=True)[f'churn_{H}'].agg(['size', 'mean']).unstack()
fl.columns = [f'{b} · {a}'.replace('size', 'spells').replace('mean', 'churn rate') for a, b in fl.columns]
fl.style.format({c: '{:.1%}' for c in fl.columns if 'rate' in c} | {c: '{:,.0f}' for c in fl.columns if 'spells' in c})

In [ ]:
# 8 · Churn rate by stable seat count, per horizon (table + chart)
tab = []
for H in HORIZONS:
    s = spells[spells[f'obs_{H}']]
    for b, gg in s.groupby('band', observed=True):
        k, n = int(gg[f'churn_{H}'].sum()), len(gg); lo, hi = wilson(k, n)
        tab.append({'horizon': H, 'stable seats': b, 'spells': n, 'clients': gg.client.nunique(), 'churned out': k, 'rate': k / n, 'lo': lo, 'hi': hi})
tab = pd.DataFrame(tab)

fig, ax = plt.subplots(figsize=(9, 4.8))
xs = np.arange(len(BANDS))
for i, H in enumerate(HORIZONS):
    t = tab[tab.horizon == H].set_index('stable seats').reindex([b[2] for b in BANDS])
    off = (i - 1.5) * 0.08
    ax.errorbar(xs + off, t['rate'], yerr=[t['rate'] - t['lo'], t['hi'] - t['rate']], marker='o', capsize=3, color=H_COLOUR[H], label=f'within {H} months')
ax.set_xticks(xs, [b[2] for b in BANDS]); ax.set_xlabel('Seat count held for 3 months (anchor)'); ax.set_ylabel('Share that churned out')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.set_ylim(0, None); ax.grid(axis='y', alpha=.3)
ax.set_title('Churn-out rate after a 3-month stable seat count, by seat count'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

wide = tab.assign(cell=lambda d: d.apply(lambda r: f"{r.rate:.0%} ({r['churned out']}/{r.spells})", axis=1)).pivot(index='stable seats', columns='horizon', values='cell').reindex([b[2] for b in BANDS])
wide.columns = [f'within {h} months' for h in wide.columns]
wide

In [ ]:
# 9 · Seat-count distributions: churned vs stayed (cumulative share of spells at or below each seat count)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), sharey=True)
for ax, H in zip(axes, HORIZONS):
    s = spells[spells[f'obs_{H}']]
    for lbl, val, col in [('churned out', True, '#E34678'), ('stayed', False, '#007AFF')]:
        x = np.sort(s.loc[s[f'churn_{H}'] == val, 'seats'].to_numpy())
        ax.step(x, np.arange(1, len(x) + 1) / len(x), where='post', color=col, label=f'{lbl} (n={len(x):,})')
    k = int(sep.loc[sep.horizon == H, 'best line: seats ≤'].iloc[0])
    ax.axvline(k + 0.5, color='#737C90', ls='--', lw=1); ax.text(k + 0.7, 0.05, f'≤ {k}', color='#737C90')
    ax.set_xscale('log'); ax.set_xlim(0.9, 200); ax.set_xticks([1, 2, 5, 10, 20, 50, 100], ['1', '2', '5', '10', '20', '50', '100']); ax.minorticks_off(); ax.set_title(f'Within {H} months'); ax.set_xlabel('Stable seat count (log)')
    ax.grid(alpha=.3); ax.legend(frameon=False, fontsize=8, loc='lower right')
axes[0].set_ylabel('Share of spells at or below'); axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
plt.suptitle('Where the two groups separate: dashed line = best dividing line', y=1.02); plt.tight_layout(); plt.show()

## Robustness

The same test with the setup changed one way at a time:
- **First spell per client only**: removes the weight of clients with many stable spells.
- **Stable = within ±1 seat**: a looser definition of "stable" (max − min ≤ 1 over 3 month-ends). The anchor is the first month it holds.
- **Anchors from 2021**: recent years only.
- **Any drop to 0 counts as churn**: no 3-month confirmation.

In [ ]:
# 10 · Robustness checks
def loose_spells(p, tol=1):
    out = []
    for client, s in p.groupby('client', sort=False):
        a = s['seats']
        ok = (a.rolling(STABLE_MONTHS).max() - a.rolling(STABLE_MONTHS).min() <= tol) & (a.rolling(STABLE_MONTHS).min() > 0)
        out.append(s[ok & ~ok.shift(fill_value=False)])
    sp = pd.concat(out); return sp[sp['month'] >= ANCHOR_FROM]

variants = {
    'Main (exact, confirmed exit)': spells,
    'First spell per client': spells.sort_values('month').groupby('client').head(1),
    'Stable = within ±1 seat': loose_spells(panel),
    'Anchors from 2021': spells[spells['month'] >= pd.Period('2021-01', 'M')],
}
p2 = panel.copy()
nxt_any = []
for client, s in p2.groupby('client', sort=False):
    tt, ex = s['t'].to_numpy(), ((s['seats'] == 0) & (s['seats'].shift().fillna(0) > 0)).to_numpy()
    out = np.full(len(tt), np.nan); up = np.nan
    for i in range(len(tt) - 1, -1, -1):
        out[i] = up - tt[i] if not np.isnan(up) else np.nan
        if ex[i]: up = tt[i]
    nxt_any.append(out)
p2['months_to_exit'] = np.concatenate(nxt_any)
for H in HORIZONS: p2[f'churn_{H}'] = p2['months_to_exit'] <= H
variants['Any drop to 0 counts as churn'] = p2.loc[spells.index]

rob = []
for name, sp in variants.items():
    for H in HORIZONS:
        r = separation(sp, H, boot=200)
        rob.append({'variant': name, 'horizon': H, 'spells': r['spells'], 'churn rate': r['churn rate'], 'AUC': r['AUC'],
                    'AUC 95% range': r['AUC 95% range'], 'best line: seats ≤': r['best line: seats ≤'],
                    'churn ≤ line': r['churn rate at or below'], 'churn above': r['churn rate above']})
rob = pd.DataFrame(rob)
rob.style.format({'churn rate': '{:.1%}', 'AUC': '{:.2f}', 'churn ≤ line': '{:.1%}', 'churn above': '{:.1%}'}).hide(axis='index')

## Does stability itself matter?

The question compares clients by their stable seat count. A related check: at the same size, is a client that has held its seat count for 3+ months less likely to churn out than one whose count changed in the last 3 months? This uses every active client-month from the anchor start, not just spell anchors, with the 12-month horizon.

In [ ]:
# 11 · Stable (held 3+ months) vs changed in the last 3 months, same size band, 12-month horizon
H = 12
cm = panel[(panel['seats'] > 0) & (panel['month'] >= ANCHOR_FROM) & panel[f'obs_{H}']].copy()
cm['band'] = pd.Categorical(cm['seats'].map(band), [b[2] for b in BANDS], ordered=True)
cm['held'] = np.where(cm['run'] >= STABLE_MONTHS, 'held 3+ months', 'changed in last 3 months')
st = cm.groupby(['band', 'held'], observed=True).agg(client_months=(f'churn_{H}', 'size'), churn_rate=(f'churn_{H}', 'mean')).unstack()
st.columns = [f'{a} · {b}' for a, b in st.columns]
st['ratio (changed ÷ held)'] = st['churn_rate · changed in last 3 months'] / st['churn_rate · held 3+ months']
st.style.format({c: '{:.1%}' for c in st.columns if 'rate' in c} | {c: '{:,.0f}' for c in st.columns if 'months' in c and 'rate' not in c} | {'ratio (changed ÷ held)': '{:.2f}×'})

## Findings

_Filled in after the run: see the next cell._

# Part 2 · By client size: under 5, 5+, 10+, 15+, 20+

The same stable spells as Part 1, split by the seat count held at the anchor. The groups overlap on purpose: 5+ includes 10+, 15+ and 20+. Each group answers "for clients holding at least this many seats".

1. **How often each group churns out** at 9, 12, 15 and 18 months.
2. **Who stays and who leaves** within each group. Leavers are compared with stayers on what the engagement data shows at the anchor: tenure, recent seat trend, distance below peak, staff turnover, hiring, how long current staff have been in their seats, and previous exits.
3. **The client list.** One row per client, written to a local file outside the repo. The notebook shows counts only.

A group needs at least 20 leavers before its profile is read. Below that it is shown but marked too few.

In [ ]:
# 12 · Size groups and client features at each anchor (engagement data only)
GROUPS = [('Under 5', 1, 4), ('5+', 5, 10**6), ('10+', 10, 10**6), ('15+', 15, 10**6), ('20+', 20, 10**6)]
G_COLOUR = {'Under 5': '#007AFF', '5+': '#00AB9C', '10+': '#FFC426', '15+': '#F46600', '20+': '#4121A9'}
MIN_LEFT = 20

pf = panel[['client', 'month', 't', 'seats', 'exit']].copy()
ev = events.groupby(['client', 'month', 'event']).size().unstack(fill_value=0)
pf = pf.merge(ev, left_on=['client', 'month'], right_index=True, how='left').fillna({'seat start': 0, 'seat end': 0})
g = pf.groupby('client', sort=False)
first_active = pf[pf['seats'] > 0].groupby('client')['t'].min()
pf['tenure_years'] = (pf['t'] - pf['client'].map(first_active)) / 12
seats_6m = g['seats'].shift(6)
pf['change_6m'] = np.where(seats_6m > 0, (pf['seats'] - seats_6m) / seats_6m, np.nan)
pf['below_peak'] = 1 - pf['seats'] / g['seats'].cummax().replace(0, np.nan)
avg12 = g['seats'].transform(lambda s: s.rolling(12, min_periods=1).mean()).replace(0, np.nan)
pf['staff_exits_12m'] = g['seat end'].transform(lambda s: s.rolling(12, min_periods=1).sum()) / avg12
pf['staff_starts_12m'] = g['seat start'].transform(lambda s: s.rolling(12, min_periods=1).sum()) / avg12
pf['prior_exits'] = g['exit'].cumsum()

sp = spells.drop(columns=['exit']).merge(pf[['client', 'month', 'tenure_years', 'change_6m', 'below_peak', 'staff_exits_12m', 'staff_starts_12m', 'prior_exits']],
                                          on=['client', 'month'], how='left')
# how long the people currently in the seats have been there (months), at the anchor month-end
a = sp[['client', 'month']].assign(me=lambda d: d['month'].dt.to_timestamp(how='end').dt.normalize()).merge(eng[['client', 'start', 'end']], on='client')
a = a[(a['start'] <= a['me']) & (a['end'].isna() | (a['end'] >= a['me']))]
a['ten'] = (a['me'] - a['start']).dt.days / 30.44
sp = sp.merge(a.groupby(['client', 'month'])['ten'].mean().rename('staff_tenure_m').reset_index(), on=['client', 'month'], how='left')

FEATURES = {
    'seats': 'Seat count held',
    'tenure_years': 'Years as a client',
    'change_6m': 'Seat change vs 6 months earlier',
    'below_peak': 'Share below the client\'s peak',
    'staff_exits_12m': 'Staff exits, last 12 months ÷ avg seats',
    'staff_starts_12m': 'Staff starts, last 12 months ÷ avg seats',
    'staff_tenure_m': 'Avg months current staff in seat',
    'prior_exits': 'Previous full exits',
}
in_group = lambda d, lo, hi: d[(d['seats'] >= lo) & (d['seats'] <= hi)]
print(f"{len(sp):,} spells with features · blank 'seat change vs 6 months' = client under 6 months old or had 0 seats then: {sp['change_6m'].isna().sum():,}")

In [ ]:
# 13 · Churn-out rate by size group and horizon
rows = []
for name, lo, hi in GROUPS:
    for H in HORIZONS:
        s = in_group(sp[sp[f'obs_{H}']], lo, hi)
        k, n = int(s[f'churn_{H}'].sum()), len(s); l, u = wilson(k, n)
        rows.append({'group': name, 'horizon': H, 'spells': n, 'clients': s['client'].nunique(), 'left': k, 'stayed': n - k, 'rate': k / n if n else np.nan, 'lo': l, 'hi': u})
gr = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 4.6))
w = 0.18
for i, H in enumerate(HORIZONS):
    t = gr[gr['horizon'] == H].set_index('group').reindex([x[0] for x in GROUPS])
    xs = np.arange(len(GROUPS)) + (i - 1.5) * w
    ax.bar(xs, t['rate'], w, color=H_COLOUR[H], label=f'within {H} months')
    ax.errorbar(xs, t['rate'], yerr=[t['rate'] - t['lo'], t['hi'] - t['rate']], fmt='none', ecolor='#50505C', capsize=2, lw=1)
ax.set_xticks(np.arange(len(GROUPS)), [x[0] for x in GROUPS]); ax.set_xlabel('Seat count held for 3 months'); ax.set_ylabel('Share that churned out')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.grid(axis='y', alpha=.3); ax.legend(frameon=False, ncol=4)
ax.set_title('Churn-out rate by size group (groups overlap: 5+ includes 10+, 15+, 20+)'); plt.tight_layout(); plt.show()

wide = gr.assign(cell=lambda d: d.apply(lambda r: f"{r.rate:.1%} · {r.left} left / {r.stayed} stayed", axis=1)).pivot(index='group', columns='horizon', values='cell').reindex([x[0] for x in GROUPS])
wide.columns = [f'within {h} months' for h in wide.columns]
wide.insert(0, 'clients (12 m)', gr[gr.horizon == 12].set_index('group')['clients'])
wide

### Who stays and who leaves, within each group (12-month horizon)

For each feature, the **median for clients who stayed vs clients who left**, and an AUC showing how well that one feature alone separates them. 0.5 = no separation, about 0.7 = useful. ↑ means higher values go with leaving; ↓ means lower values go with leaving. Groups with fewer than 20 leavers are greyed out in the chart and marked in the table.

In [ ]:
# 14 · Feature profile: stayed vs left, per group, 12-month horizon
H = 12
prof, heat, arrow = [], {}, {}
for name, lo, hi in GROUPS:
    s = in_group(sp[sp[f'obs_{H}']], lo, hi)
    y = s[f'churn_{H}'].astype(int)
    few = y.sum() < MIN_LEFT
    for f, label in FEATURES.items():
        m = s[f].notna()
        yy, xx = y[m], s.loc[m, f]
        auc = roc_auc_score(yy, xx) if (yy.nunique() == 2 and xx.nunique() > 1) else np.nan
        p = mannwhitneyu(xx[yy == 1], xx[yy == 0]).pvalue if (yy.sum() > 0 and (yy == 0).sum() > 0 and xx.nunique() > 1) else np.nan
        strength = max(auc, 1 - auc) if pd.notna(auc) else np.nan
        prof.append({'group': name, 'feature': label, 'leavers': int(yy.sum()), 'median · stayed': xx[yy == 0].median(), 'median · left': xx[yy == 1].median(),
                     'AUC': strength, 'direction': ('↑' if auc > 0.5 else '↓') if pd.notna(auc) else '', 'p': p, 'too few leavers': few})
        heat[(name, label)] = strength; arrow[(name, label)] = (('↑' if auc > 0.5 else '↓') if pd.notna(auc) else '')
prof = pd.DataFrame(prof)

# heatmap: separation strength per feature (rows) and group (columns)
gnames, fnames = [x[0] for x in GROUPS], list(FEATURES.values())
M = np.array([[heat[(gname, f)] for gname in gnames] for f in fnames], dtype=float)
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(M, cmap='Blues', vmin=0.5, vmax=0.8, aspect='auto')
few_groups = set(prof.loc[prof['too few leavers'], 'group'])
for i, f in enumerate(fnames):
    for j, gname in enumerate(gnames):
        v = M[i, j]
        txt = '–' if np.isnan(v) else f"{v:.2f} {arrow[(gname, f)]}"
        ax.text(j, i, txt, ha='center', va='center', fontsize=9, color=('#9AA3B5' if gname in few_groups else ('white' if v > 0.68 else '#1B1F2A')))
for j, gname in enumerate(gnames):
    if gname in few_groups:
        ax.add_patch(plt.Rectangle((j - .5, -.5), 1, len(fnames), color='white', alpha=.6))
ax.set_xticks(range(len(gnames)), [f"{gname}\n({int(prof[prof.group == gname].leavers.max())} left)" for gname in gnames]); ax.set_yticks(range(len(fnames)), fnames)
ax.set_title('How well each feature separates leavers from stayers (AUC, 12 months)\n↑ higher = more likely to leave · ↓ lower = more likely to leave · faded = under 20 leavers')
plt.colorbar(im, ax=ax, shrink=.8); plt.tight_layout(); plt.show()

PCT = {FEATURES[k] for k in ['change_6m', 'below_peak', 'staff_exits_12m', 'staff_starts_12m']}
fmt = lambda v, f: '–' if pd.isna(v) else (f'{v:+.0%}' if f == FEATURES['change_6m'] else f'{v:.0%}' if f in PCT else f'{v:,.1f}')
out = prof.assign(**{'median · stayed': [fmt(v, f) for v, f in zip(prof['median · stayed'], prof['feature'])], 'median · left': [fmt(v, f) for v, f in zip(prof['median · left'], prof['feature'])],
                     'AUC': prof['AUC'].map(lambda v: '–' if pd.isna(v) else f'{v:.2f}'), 'p': prof['p'].map(lambda v: '–' if pd.isna(v) else f'{v:.1e}')})
out['note'] = np.where(out['too few leavers'], 'too few leavers', '')
out.drop(columns=['too few leavers']).set_index(['group', 'feature'])

In [ ]:
# 15 · Churn rate by profile, per group (12-month horizon): the readable version of the heatmap
H = 12
CUTS = {
    'Years as a client': ('tenure_years', [-0.01, 1, 2, 4, 100], ['under 1 year', '1–2 years', '2–4 years', '4+ years']),
    'Seats vs 6 months earlier': ('change_6m', [-10, -0.0001, 0.0001, 100], ['shrank', 'same', 'grew']),
    'Below peak seat count': ('below_peak', [-0.01, 0.0001, 0.25, 1.01], ['at peak', 'up to 25% below', 'more than 25% below']),
    'Staff exits last 12 months ÷ avg seats': ('staff_exits_12m', [-0.01, 0.0001, 0.5, 100], ['none', 'up to half', 'more than half']),
    'Avg months current staff in seat': ('staff_tenure_m', [-1, 6, 12, 24, 10**4], ['under 6', '6–12', '12–24', '24+']),
}
rows = []
for title, (col, bins, labels) in CUTS.items():
    for name, lo, hi in GROUPS:
        s = in_group(sp[sp[f'obs_{H}']], lo, hi)
        b = pd.cut(s[col], bins, labels=labels)
        if col == 'change_6m':
            b = b.cat.add_categories('new (under 6 months)').fillna('new (under 6 months)')
        for lab, gg in s.groupby(b, observed=True):
            rows.append({'profile': title, 'level': lab, 'group': name, 'cell': f"{gg[f'churn_{H}'].mean():.0%} ({int(gg[f'churn_{H}'].sum())}/{len(gg)})"})
bt = pd.DataFrame(rows).pivot_table(index=['profile', 'level'], columns='group', values='cell', aggfunc='first', sort=False)[[x[0] for x in GROUPS]]
bt.fillna('–')

In [ ]:
# 16 · Client list: each client's latest stable spell, with its outcome at each horizon (written outside the repo)
OUT_DIR = Path.home() / 'Desktop/Projects/Churn Analysis/Outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
names = raw.dropna(subset=['Shortcode']).drop_duplicates('Shortcode').set_index('Shortcode')['Client Name']
latest = sp.sort_values('month').groupby('client').tail(1).copy()
for H in HORIZONS:
    latest[f'within {H} months'] = np.where(~latest[f'obs_{H}'], 'too recent to tell', np.where(latest[f'churn_{H}'], 'left', 'stayed'))
for name, lo, hi in GROUPS:
    latest[f'in {name}'] = latest['seats'].between(lo, hi)
latest['seats now (Oct 2026)'] = latest['client'].map(panel[panel['month'] == LAST_MONTH].set_index('client')['seats']).fillna(0).astype(int)
cols = ['client', 'client name', 'month', 'seats', 'seats now (Oct 2026)'] + [f'in {x[0]}' for x in GROUPS] + [f'within {H} months' for H in HORIZONS] + list(FEATURES)[1:]
latest = latest.assign(**{'client name': latest['client'].map(names), 'month': latest['month'].astype(str)})[cols] \
    .rename(columns={'client': 'shortcode', 'month': 'anchor month', 'seats': 'stable seats'} | {k: v for k, v in FEATURES.items() if k != 'seats'})
path = OUT_DIR / f"stable_seat_clients_{pd.Timestamp.today():%Y-%m-%d}.csv"
latest.to_csv(path, index=False)
print(f"Wrote {len(latest):,} clients to {path}")
summ = pd.DataFrame({x[0]: latest.loc[latest[f'in {x[0]}'], 'within 12 months'].value_counts() for x in GROUPS}).reindex(['stayed', 'left', 'too recent to tell']).fillna(0).astype(int)
summ.index.name = "latest spell, 12 months"
summ

## Findings · Part 2 (by size group)

**How often each group churns out.** Under-5 clients churn out at about 3 times the rate of every 5+ group. Above 5 seats the rate is roughly flat until 20+.

| Held for 3 months | Clients | Within 9 m | Within 12 m | Within 15 m | Within 18 m |
|---|---|---|---|---|---|
| Under 5 | 1,571 | 22% | 26% | 30% | 34% |
| 5+ | 355 | 6% | 9% | 11% | 14% |
| 10+ | 149 | 5% | 8% | 9% | 12% |
| 15+ | 89 | 6% | 8% | 11% | 14% |
| 20+ | 56 | 3% | 5% | 6% | 7% |

The 10+, 15+ and 20+ ranges overlap: the differences between them are within noise.

**Who leaves and who stays (12-month horizon).**

- **Under 5** (753 leavers).
  - Leavers: **1 seat**, **new** (under a year: 29% leave vs 17–21% for older clients), **shrinking** (33% leave vs 8% for clients that grew in the last 6 months), staff in their seats **under 6 months** (30%), and **more than half the staff replaced** in the last year (35%).
  - Stayers: growing, with staff who have been in their seats 6+ months.
  - Strongest single signal: seat change vs 6 months earlier (AUC 0.72). It only applies to clients at least 6 months old.
- **5+** (106 leavers). Nothing separates leavers strongly; the best single feature has AUC 0.63.
  - Leavers lean **new** (21% leave in their first 6 months vs 6–14% after) and **shrinking** (14% vs 6% for growing).
  - Seat count within 5+ doesn't matter (AUC 0.52).
- **10+** (39 leavers).
  - **Tenure** matters most (AUC 0.64): 12–14% leave within their first 2 years vs 5–6% after.
  - **Staff turnover**: 19% leave when more than half the staff were replaced in the last year, vs 6–7% otherwise.
  - Shrinking clients leave at 13%, growing ones at 6%.
- **15+** (23 leavers, just enough to read).
  - **Staff turnover** is the clearest signal (AUC 0.74): 28% leave when more than half the staff were replaced in the last year, vs 0–8% otherwise.
  - The **smaller clients in the group** leave more: median 17 seats for leavers vs 23 for stayers.
  - So does **tenure**: 14% under 2 years vs 4% at 4+ years.
- **20+** (8 leavers). Too few to profile. It has the lowest churn of any group. The same patterns appear (high staff turnover, shorter tenure, below peak), but 8 leavers can't confirm them.

**The common thread.** Clients that leave tend to be **new, shrinking, and replacing their staff**. Clients that stay tend to be **established, growing or steady, and keep their staff**. For under-5 clients, holding just **1 seat** adds to that. Only two single signals reach the useful level (AUC ≥ 0.7): seat change in under-5 clients and staff turnover in 15+ clients. Combining the features into one score may separate leavers better than any single feature does. That isn't tested here.

**Client list.** Each client's latest stable spell and its outcome at each horizon are in `Desktop/Projects/Churn Analysis/Outputs/stable_seat_clients_<date>.csv` (1,995 clients), outside the repo. Many recent spells can't be judged yet: 501 under-5 clients and 196 5+ clients are too recent for a 12-month outcome.

# Part 3 · The role behind the seat, and clients that grow from 1

Two questions, both about clients with a single seat:

- **A. 1-seat clients that churn.** When a client has held exactly 1 seat for 3 months (a 1-seat stable spell from Part 1), is the **JP title or JP category** of that one person related to whether the client churns out within 12 months?
- **B. Clients that start with 1 and grow.** Of the clients that held exactly 1 seat through their first 6 months, how many go on to 2, 3–4, 5+ seats, how long does that take, and what do the growers look like early on?

**Role source.** The engagement list has no job titles, so roles come from the Staff Central *Staff Report with Details* (16 Nov 2026), cleaned and backfilled with [`shared/jp_titles.py`](../shared/jp_titles.py). It strips seniority words (Senior, Level 2, II) to give one **title base**, and fills blank titles and categories from the `Designation (All)` field only where the match is unambiguous.

**Caveat, read first.** The staff report has **one row per person**, holding their *latest* engagement. A seat's role therefore comes from either:
- **engagement**: the anchor engagement is that person's latest one, so the role is the role in that seat; or
- **person**: the person has moved on, so the role is their role *today*, possibly at another client.

Which of these applies is itself tied to the outcome (step 18), so every test below accounts for it.

In [ ]:
# 17 · JP titles: load the Staff Central staff report, then clean and backfill titles with shared/jp_titles.py
import sys
import statsmodels.formula.api as smf
from scipy.stats import chi2

REPO_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').is_file())
JP_DIR = next(d for d in [REPO_ROOT / 'reference_code' / 'JP_titles_cat_cleaning', REPO_ROOT / 'shared'] if (d / 'jp_titles.py').is_file())
sys.path.insert(0, str(JP_DIR))                 # jp_titles.py moved from shared/ to reference_code/JP_titles_cat_cleaning/
from jp_titles import backfill_jp

STAFF_REPORT = Path.home() / 'Desktop/Projects/Data/Top_roles_APAC_adhoc/Staff Report with Details 16-Nov-2026 [CONFIDENTIAL] (1).xlsx'
sr_raw = pd.read_excel(STAFF_REPORT, dtype=str)
sr, jp_reports = backfill_jp(sr_raw)          # strict cascade: ambiguous keys stay blank
print(f"{len(sr_raw):,} staff report rows ({sr_raw['CSID'].nunique():,} people). "
      f"JP title base: {sr_raw['JP Title (All)'].notna().mean():.0%} populated in the source, {sr['JP Title Base (Filled)'].notna().mean():.0%} after backfill. "
      f"JP category: {sr_raw['JP Category (All)'].notna().mean():.0%} → {sr['JP Category (Filled)'].notna().mean():.0%}.")
pd.concat({'JP title base': jp_reports['title_source_counts'], 'JP category': jp_reports['category_source_counts']}, axis=1).fillna(0).astype(int)

In [ ]:
# 18 · The one seat behind each 1-seat stable spell, and its JP title / category
eng_csid = eng.merge(raw[['Engagement ID', 'CSID']].drop_duplicates('Engagement ID'), on='Engagement ID', how='left')
by_eid = sr.dropna(subset=['Engagement ID']).drop_duplicates('Engagement ID').set_index('Engagement ID')
by_csid = sr.drop_duplicates('CSID', keep='last').set_index('CSID')

def attach_role(d):
    # d has one engagement per row; adds role_match ('engagement' / 'person'), jp_title, jp_category
    d = d.copy()
    d['role_match'] = np.where(d['Engagement ID'].isin(by_eid.index), 'engagement', 'person')
    for col, new in [('JP Title Base (Filled)', 'jp_title'), ('JP Category (Filled)', 'jp_category')]:
        d[new] = d['Engagement ID'].map(by_eid[col]).where(d['role_match'].eq('engagement'), d['CSID'].map(by_csid[col])).fillna('Unknown')
    return d

one = spells[spells['seats'] == 1].copy()
one['month_end'] = one['month'].dt.to_timestamp(how='end').dt.normalize()
one = one.merge(eng_csid, on='client')
one = one[(one['start'] <= one['month_end']) & (one['end'].isna() | (one['end'] >= one['month_end']))]
assert not one.duplicated(['client', 'month']).any() and len(one) == (spells['seats'] == 1).sum()
one = attach_role(one)
one['anchor_year'] = one['month'].dt.year

H = 12
o1 = one[one[f'obs_{H}']].sort_values('month').copy()
o1['y'] = o1[f'churn_{H}'].astype(int)
print(f"{len(o1):,} 1-seat stable spells with 12 months observed, from {o1.client.nunique():,} clients. 12-month churn-out rate {o1.y.mean():.0%}.")
cov = (o1.groupby(o1['y'].map({0: 'stayed', 1: 'churned out'}))
         .agg(spells=('y', 'size'),
              **{'role from the seat itself (engagement)': ('role_match', lambda s: s.eq('engagement').mean()),
                 'JP category unknown': ('jp_category', lambda s: s.eq('Unknown').mean()),
                 'JP title unknown': ('jp_title', lambda s: s.eq('Unknown').mean())}))
for c in cov.columns[1:]:
    cov[c] = cov[c].map('{:.0%}'.format)
cov

**Read this before the role tables.** For clients that stayed, two-thirds of the seats can be matched to their exact engagement. For clients that churned out, only a third can: the person was usually moved to another client or left, so their staff-report row is a later engagement. Churned-out seats are also far more often **Unknown** (no JP data at all). So:
- **Unknown** is mostly a sign that the person and record are gone. It is a by-product of the outcome, not a role. It is shown but not read as a role effect.
- Comparisons between categories hold the match type and the anchor year constant (step 20).

In [ ]:
# 19 · 12-month churn-out rate by the JP category of the one seat (categories with 15+ spells; the rest grouped)
MIN_N = 15
vc = o1['jp_category'].value_counts()
OTHER = f'Other (each under {MIN_N} spells)'
o1['cat_group'] = np.where(o1['jp_category'].eq('Unknown'), 'Unknown',
                  np.where(o1['jp_category'].map(vc) >= MIN_N, o1['jp_category'], OTHER))

def rate_cell(g):
    return f"{g.y.mean():.0%} ({int(g.y.sum())}/{len(g)})" if len(g) else '–'

rows = []
for c, gg in o1.groupby('cat_group'):
    k, n = int(gg.y.sum()), len(gg); lo, hi = wilson(k, n)
    rows.append({'JP category': c, 'spells': n, 'churned out': k, 'rate': k / n, 'lo': lo, 'hi': hi,
                 'role from the seat (engagement)': rate_cell(gg[gg.role_match == 'engagement']),
                 'role from the person today': rate_cell(gg[gg.role_match == 'person'])})
cat_tab = pd.DataFrame(rows).sort_values('rate')
known_rate = o1.loc[o1.cat_group.ne('Unknown'), 'y'].mean()

plot = cat_tab.set_index('JP category')
fig, ax = plt.subplots(figsize=(9, 0.38 * len(plot) + 1.2))
ys = np.arange(len(plot))
col = ['#8E8E93' if c in ('Unknown', OTHER) else '#007AFF' for c in plot.index]
for y_, (_, r), c_ in zip(ys, plot.iterrows(), col):
    ax.errorbar(r['rate'], y_, xerr=[[r['rate'] - r['lo']], [r['hi'] - r['rate']]], fmt='none', ecolor=c_, capsize=3, alpha=.7)
ax.scatter(plot['rate'], ys, c=col, zorder=3)
ax.axvline(known_rate, color='#F46600', ls='--', lw=1, label=f'all known categories ({known_rate:.0%})')
ax.set_yticks(ys, [f"{c}  (n={n})" for c, n in zip(plot.index, plot['spells'])])
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.set_xlim(0, 1); ax.grid(axis='x', alpha=.3)
ax.set_xlabel('Share of 1-seat clients that churned out within 12 months (95% interval)')
ax.set_title('1-seat clients: 12-month churn-out by the JP category of the seat'); ax.legend(frameon=False, loc='lower right')
plt.tight_layout(); plt.show()

out = cat_tab.assign(**{'all': cat_tab.apply(lambda r: f"{r.rate:.0%} ({r['churned out']}/{r.spells})", axis=1)})
out[['JP category', 'all', 'role from the seat (engagement)', 'role from the person today']].set_index('JP category')

In [ ]:
# 20 · Does JP category add anything once anchor year and match type are held constant?
#      Logistic regression with and without category; likelihood-ratio test; AUC (0.5 = coin flip, ~0.7 = useful).
def lr_test(d, label, term='C(cat_group)'):
    base = smf.logit('y ~ C(role_match) + C(anchor_year)', d).fit(disp=0, maxiter=200)
    full = smf.logit(f'y ~ C(role_match) + C(anchor_year) + {term}', d).fit(disp=0, maxiter=200)
    lr = 2 * (full.llf - base.llf); dfree = int(full.df_model - base.df_model)
    return {'sample': label, 'spells': len(d), 'churned out': int(d.y.sum()), 'categories compared': d['cat_group'].nunique(),
            'p-value (category adds nothing)': round(chi2.sf(lr, dfree), 3),
            'AUC: year + match type': round(roc_auc_score(d.y, base.predict()), 3),
            'AUC: + category': round(roc_auc_score(d.y, full.predict()), 3)}, full

known = o1[o1.cat_group.ne('Unknown')]
res = [lr_test(o1, 'all spells, Unknown included'), lr_test(known, 'known category only'),
       lr_test(known.drop_duplicates('client'), 'known category, first 1-seat spell per client')]
display(pd.DataFrame([r[0] for r in res]).set_index('sample'))

# odds ratios against Accounting (the largest category), known categories only
m_known = smf.logit("y ~ C(role_match) + C(anchor_year) + C(cat_group, Treatment('Accounting'))", known).fit(disp=0, maxiter=200)
ci = m_known.conf_int()
orr = pd.DataFrame({'odds ratio vs Accounting': np.exp(m_known.params), 'low': np.exp(ci[0]), 'high': np.exp(ci[1]), 'p': m_known.pvalues})
orr = orr[orr.index.str.contains('cat_group')]
orr.index = orr.index.str.extract(r'\[T\.(.*)\]')[0].values
print(f"Odds ratios vs Accounting (held constant: year, match type). With {len(orr)} comparisons, about 1 would show p < 0.05 by chance alone.")
orr.sort_values('odds ratio vs Accounting').round(2)

In [ ]:
# 21 · The same view by JP title base (titles with 10+ spells), known titles only
MIN_T = 10
kt = o1[o1.jp_title.ne('Unknown')]
tv = kt['jp_title'].value_counts()
tt = (kt[kt.jp_title.map(tv) >= MIN_T].groupby('jp_title')
        .apply(lambda g: pd.Series({'main JP category': g.jp_category[g.jp_category.ne('Unknown')].mode().iat[0] if g.jp_category.ne('Unknown').any() else 'Unknown',
                                    'spells': len(g), 'rate': g.y.mean(), 'churned out (of spells)': rate_cell(g),
                                    'role from the seat (engagement)': rate_cell(g[g.role_match == 'engagement']),
                                    'role from the person today': rate_cell(g[g.role_match == 'person'])}), include_groups=False)
        .sort_values('rate'))
kt2 = kt.assign(cat_group=np.where(kt.jp_title.map(tv) >= MIN_T, kt.jp_title, 'Other title'))
r_t = lr_test(kt2, 'known title, titles with 10+ spells (rest grouped)')[0]
print(f"{kt.jp_title.nunique()} distinct title bases among {len(kt):,} spells with a known title; {(tv >= MIN_T).sum()} have {MIN_T}+ spells. "
      f"Test of title, year and match type held constant: p = {r_t['p-value (category adds nothing)']}, "
      f"AUC {r_t['AUC: year + match type']} → {r_t['AUC: + category']}. Overall rate for known titles {kt.y.mean():.0%}.")
tt.drop(columns='rate')

## Findings · Part 3A (role of the one seat)

**Short answer: no meaningful relation.** What the one person does doesn't tell you whether a 1-seat client will churn. Being a 1-seat client is the risk (Part 1). Within 1-seat clients, the early signals are shrinking, newness and staff turnover (Part 2), not the role.

- **Base.** 1,417 1-seat stable spells (1,234 clients) with 12 months observed. 38% churned out within 12 months.
- **The raw table looks like it has a pattern.** Churn ranges from 13% (Engineering Civil, n=15) to 65% (Recruitment, n=20). The two largest categories sit near the middle: Accounting 26% (n=212) and Virtual Assistant 32% (n=122).
- **Most of that pattern is data, not role.**
  - Seats with **no JP data (Unknown)** churn at 50%, vs 32% for known roles. Unknown is far more common for churned clients (43% vs 26% of seats) because the person or their record is gone.
  - Seats are also matched differently depending on the outcome: by their own engagement for 67% of stayers but only 32% of churners.
- **Once both are accounted for** (Unknown left out, with anchor year and match type held constant), JP category adds nothing measurable:
  - category: p = 0.47, AUC 0.794 → 0.799;
  - first 1-seat spell per client: p = 0.24;
  - JP title base (21 titles with 10+ spells): p = 0.19.
- **Two hints, not findings.** **Recruitment** (13/20 churned, odds 3.5× Accounting, p = 0.03) and **Sales Professional** (23/46, 2.1×, p = 0.05; Business Development Manager 10/15). With 14 comparisons, about one would reach p < 0.05 by chance, and most of these churners have a person-level role match. They are worth watching if more data comes in.
- **Where the role is certain** (the seat matches its own engagement), most categories churn at 7–27%, with no clear ordering.

## Part 3B · Clients that start with 1 seat: who grows?

**Cohort.** Clients whose **first seat started in Jan 2016 or later** and who held **exactly 1 seat at each of their first 6 month-ends** (month 1 = the month of their first seat). Clients that dropped to 0, or that already had 2+ seats in those 6 months, are not in the cohort; step 22 shows how many there are.

**What is measured.** From month 7 on: the **peak seat count reached** by month 12, 24, 36 and 48. Each milestone uses only clients old enough to have reached it, so the samples get smaller and older as the milestone gets later. Churn out uses the same confirmed-exit rule as Parts 1–2.

In [ ]:
# 22 · How clients start: seat count over their first 6 month-ends (first seat Jan 2016 or later)
COHORT_FROM = pd.Period('2016-01', 'M'); START_MONTHS = 6; MILESTONES = [12, 24, 36, 48]
first_m = panel[panel.seats > 0].groupby('client')['month'].min().rename('first_month')
pc = panel.merge(first_m, on='client')
pc = pc[pc['first_month'] >= COHORT_FROM].copy()
pc['age'] = pc['t'] - pc['first_month'].map(lambda p: p.ordinal) + 1      # month 1 = month of the first seat
pc = pc[pc['age'] >= 1]
max_age = pc.groupby('client')['age'].max()

early = pc[pc.age <= START_MONTHS].groupby('client')['seats'].agg(['min', 'max'])
early = early[max_age.reindex(early.index) >= START_MONTHS]
def start_type(r):
    if r['min'] == 0: return 'dropped to 0 within 6 months'
    if r['max'] == 1: return '1 seat at every month-end (the cohort)'
    if r['max'] == 2: return 'peak of 2 in first 6 months'
    if r['max'] <= 4: return 'peak of 3–4 in first 6 months'
    return 'peak of 5+ in first 6 months'
early['start type'] = early.apply(start_type, axis=1)
st = early['start type'].value_counts().rename('clients').to_frame()
st['share'] = (st['clients'] / st['clients'].sum()).map('{:.0%}'.format)
print(f"{len(early):,} clients with a first seat from {COHORT_FROM} and at least 6 months of data")
cohort = early.index[early['start type'].str.startswith('1 seat')]
st

In [ ]:
# 23 · Peak seat count reached by each milestone, for the start-at-1 cohort
PEAK_BINS, PEAK_LAB = [0, 1, 2, 4, 9, 10**6], ['stayed at 1', '2', '3–4', '5–9', '10+']
PEAK_COL = {'stayed at 1': '#C7C7CC', '2': '#9BC9FF', '3–4': '#007AFF', '5–9': '#00AB9C', '10+': '#4121A9'}
cp = pc[pc.client.isin(cohort)]
exit_age = cp[cp['exit']].groupby('client')['age'].min()

rows = []
for M in MILESTONES:
    ok = max_age[cohort][max_age[cohort] >= M].index
    w = cp[cp.client.isin(ok) & (cp.age <= M)]
    peak = w.groupby('client')['seats'].max()
    seats_at_M = cp[cp.client.isin(ok) & (cp.age == M)].set_index('client')['seats']
    dist = pd.cut(peak, PEAK_BINS, labels=PEAK_LAB).value_counts(normalize=True).reindex(PEAK_LAB)
    rows.append({'by month': M, 'clients observed': len(ok), **dist.to_dict(),
                 'reached 3+': (peak >= 3).mean(), 'reached 5+': (peak >= 5).mean(),
                 'churned out by then': exit_age.reindex(ok).le(M).mean(), 'still active at that month': (seats_at_M > 0).mean()})
peak_tab = pd.DataFrame(rows).set_index('by month')

fig, ax = plt.subplots(figsize=(9, 3.6))
left = np.zeros(len(peak_tab))
for lab in PEAK_LAB:
    v = peak_tab[lab].to_numpy()
    ax.barh([f'by month {m}\n(n={n})' for m, n in zip(peak_tab.index, peak_tab['clients observed'])], v, left=left, color=PEAK_COL[lab], label=lab)
    for i, (l, x) in enumerate(zip(left, v)):
        if x >= .04: ax.text(l + x / 2, i, f'{x:.0%}', ha='center', va='center', fontsize=9, color='white' if lab not in ('stayed at 1', '2') else 'black')
    left += v
ax.invert_yaxis(); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.set_xlim(0, 1)
ax.set_title('Clients with 1 seat through their first 6 months: peak seat count reached')
ax.legend(title='peak seats', frameon=False, ncol=5, loc='upper center', bbox_to_anchor=(.5, -.12))
plt.tight_layout(); plt.show()

show = peak_tab.copy()
for c in show.columns[1:]:
    show[c] = show[c].map('{:.0%}'.format)
show

In [ ]:
# 24 · When growth happens: share that has reached 2, 3 and 5 seats by each month of age
#      (fixed group: cohort clients with 48+ months of data, so every point uses the same clients)
FIX = 48
fx = max_age[cohort][max_age[cohort] >= FIX].index
w = cp[cp.client.isin(fx) & (cp.age <= FIX)].sort_values(['client', 'age'])
w['peak_so_far'] = w.groupby('client')['seats'].cummax()
ages = np.arange(1, FIX + 1)
curve = pd.DataFrame({f'reached {k}+': w.assign(r=w.peak_so_far >= k).groupby('age')['r'].mean().reindex(ages) for k in (2, 3, 5)})
curve['churned out'] = [exit_age.reindex(fx).le(a).mean() for a in ages]

fig, ax = plt.subplots(figsize=(9, 4.2))
for c, colr in zip(curve.columns, ['#9BC9FF', '#007AFF', '#00AB9C', '#F46600']):
    ax.plot(curve.index, curve[c], color=colr, lw=2, ls='--' if c == 'churned out' else '-', label=c)
    ax.annotate(f'{curve[c].iloc[-1]:.0%}', (FIX, curve[c].iloc[-1]), xytext=(4, 0), textcoords='offset points', va='center', color=colr)
ax.axvspan(1, START_MONTHS, color='#E5E5EA', alpha=.6); ax.text(START_MONTHS / 2 + .5, .62, '1 seat\n(cohort rule)', ha='center', fontsize=8)
ax.set_xlabel('Months since the first seat'); ax.set_ylabel('Share of clients'); ax.set_xlim(1, FIX + 4); ax.set_ylim(0, .7)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}')); ax.grid(alpha=.3); ax.legend(frameon=False)
ax.set_title(f'Start-at-1 clients with {FIX}+ months of data (n={len(fx)}): growth vs churn by age')
plt.tight_layout(); plt.show()

# months from the first seat to reaching 2, 3, 5 seats, among cohort clients that ever did (all cohort clients, any age)
reach = {}
for k in (2, 3, 5):
    a = cp[cp.seats >= k].groupby('client')['age'].min()
    reach[f'{k}+ seats'] = {'clients that reached it': len(a), 'share of cohort': f"{len(a) / len(cohort):.0%}",
                            'median months to get there': int(a.median()), 'middle half (months)': f"{int(a.quantile(.25))}–{int(a.quantile(.75))}"}
pd.DataFrame(reach).T

In [ ]:
# 25 · What growers look like early: timing of the 2nd seat, and the role of the first seat
#      Cohort clients with 36+ months of data. Outcome: peak of 3+ seats by month 36.
G = 36
g36 = max_age[cohort][max_age[cohort] >= G].index
w = cp[cp.client.isin(g36) & (cp.age <= G)]
peak36 = w.groupby('client')['seats'].max()
second = w[w.seats >= 2].groupby('client')['age'].min().reindex(g36)
gw = pd.DataFrame({'peak': peak36, 'second_seat_age': second})
gw['grew3'] = (gw.peak >= 3).astype(int); gw['grew5'] = (gw.peak >= 5).astype(int)
gw['churned'] = exit_age.reindex(g36).le(G).astype(int)
gw['2nd seat'] = pd.cut(gw.second_seat_age.fillna(999), [0, 12, 24, 36, 10**4], labels=['by month 12', 'month 13–24', 'month 25–36', 'not by month 36'])
t1 = gw.groupby('2nd seat', observed=True).agg(clients=('peak', 'size'), **{'reached 3+ by 36': ('grew3', 'mean'), 'reached 5+ by 36': ('grew5', 'mean'), 'churned out by 36': ('churned', 'mean')})
for c in t1.columns[1:]:
    t1[c] = t1[c].map('{:.0%}'.format)
print(f"{len(g36)} start-at-1 clients with {G}+ months of data: {gw.grew3.mean():.0%} reached 3+ seats by month {G}, {gw.grew5.mean():.0%} reached 5+.")
display(t1)

# role of the one seat at month 6 (the cohort holds exactly 1 seat then)
m6 = pc[pc.client.isin(g36) & (pc.age == START_MONTHS)][['client', 'month']]
m6['month_end'] = m6['month'].dt.to_timestamp(how='end').dt.normalize()
m6 = m6.merge(eng_csid, on='client')
m6 = m6[(m6['start'] <= m6['month_end']) & (m6['end'].isna() | (m6['end'] >= m6['month_end']))].drop_duplicates('client')
m6 = attach_role(m6).set_index('client').join(gw)
m6['first_year'] = m6['month'].dt.year
cv = m6['jp_category'].value_counts()
m6['cat_group'] = np.where(m6.jp_category.eq('Unknown'), 'Unknown', np.where(m6.jp_category.map(cv) >= MIN_N, m6.jp_category, OTHER))
t2 = (m6.groupby('cat_group').agg(clients=('grew3', 'size'), grew=('grew3', 'mean'), **{'reached 3+ by 36 (n)': ('grew3', 'sum')})
        .sort_values('grew', ascending=False))
t2['reached 3+ by 36'] = t2.apply(lambda r: f"{r.grew:.0%} ({int(r['reached 3+ by 36 (n)'])}/{int(r.clients)})", axis=1)
kn = m6[m6.cat_group.ne('Unknown')].assign(y=lambda d: d.grew3)
b0 = smf.logit('y ~ C(role_match) + C(first_year)', kn).fit(disp=0, maxiter=200)
b1 = smf.logit('y ~ C(role_match) + C(first_year) + C(cat_group)', kn).fit(disp=0, maxiter=200)
p_cat = chi2.sf(2 * (b1.llf - b0.llf), int(b1.df_model - b0.df_model))
print(f"First seat's JP category vs reaching 3+ by month {G} (known categories, year and match type held constant): p = {p_cat:.3f}, "
      f"AUC {roc_auc_score(kn.y, b0.predict()):.3f} → {roc_auc_score(kn.y, b1.predict()):.3f}")
t2[['clients', 'reached 3+ by 36']]

## Findings · Part 3B (start at 1, then grow)

**Yes, some do, but it is a minority, and it is slow.** Churn outpaces growth.

- **How clients start.** Of 2,089 clients whose first seat was in 2016 or later:
  - 903 (43%) held exactly 1 seat through their first 6 months (the cohort);
  - 18% had already dropped to 0;
  - 39% had 2+ seats.
- **How far the 1-seat starters grow** (peak seats reached; each milestone uses clients old enough to have reached it):

| By month | Clients | Still 1 | Reached 2 | 3–4 | 5–9 | 10+ | Reached 3+ | Reached 5+ | Churned out |
|---|---|---|---|---|---|---|---|---|---|
| 12 | 824 | 83% | 12% | 4% | 0% | 0% | 4% | 0% | 22% |
| 24 | 691 | 70% | 17% | 11% | 2% | 0% | 13% | 2% | 42% |
| 36 | 571 | 64% | 17% | 11% | 6% | 1% | 18% | 7% | 54% |
| 48 | 457 | 61% | 16% | 13% | 8% | 2% | 23% | 10% | 62% |

  So about **1 in 4** of the clients with 4 years of history reached 3+ seats, and **1 in 10** reached 5+. **6 in 10** never went past 1 seat, and 62% had churned out by month 48.
- **Growth is a step-by-step climb.** Among clients that got there:
  - 2nd seat: median **12 months** after the first (middle half 9–19);
  - 3 seats: **21 months** (13–32);
  - 5 seats: **31 months** (25–45).
  Almost no one jumps from 1 straight to 5.
- **The early tell is the 2nd seat.** For clients with 36+ months of data:
  - 2nd seat **within the first year**: 58% reach 3+ and 28% reach 5+ by month 36;
  - 2nd seat in **year 2**: 48% and 16%;
  - **no 2nd seat by month 36** (64% of clients): 73% had churned out.
  Even early growers can leave: 25% of first-year growers churned out by month 36.
- **The first seat's role barely matters for growth either** (p = 0.09, AUC 0.68 → 0.72 with year and match type held constant). IT Software Development first seats reached 3+ most often (7/22, 32%), and Accounting (9/76, 12%) and Real Estate (2/19, 11%) least. The numbers are small; treat this as a hint.

## Assumptions

| # | Assumption | Why |
|---|---|---|
| 1 | **Seat** = one engagement row at an external client, active on a month-end (engagement date ≤ month-end, and disengagement date blank or ≥ month-end). | The file has one row per staff engagement. |
| 2 | `Category`, `Employment Status` and `User Status` are **not** used to pick seats. | They are the same on every row for a person (100% of CSIDs), so they describe the person today, not the engagement. |
| 3 | **Internal and test accounts removed**: shortcodes starting `CS` (incl. Disengaged Staff Pool `CSTREENGAGE`, CS Now, CS departments, `CSS…` test accounts) and Cloudstaff-named accounts under other codes (`CLOUDSTAFF`, `CLOUDSTAFFINC`, `CIB107`, `CIB124`, `CIB132`, `CIB137`, `CIB139`, `OPAINTBIZ`). | They are not clients. |
| 4 | Dropped: duplicate Engagement IDs (first kept), status `bench`, disengagement before engagement, status `disengage` with no date. Counts in step 2. | Can't be placed on a timeline, or not a working seat. |
| 5 | **Client** = `Shortcode`. | One name per shortcode in this file. |
| 6 | Seat counts run to **Oct 2026**, the last complete month-end before the 15 Nov 2026 extract. | Avoids a partial month. |
| 7 | **Stable** = the same non-zero seat count at 3 consecutive month-ends. The **anchor** is the 3rd of those months. One client can have several spells. | The first point at which we know the count has held for 3 months. A ±1-seat variant is in Robustness. |
| 8 | **Churn out** = seats drop to 0 and stay 0 for 3 month-ends (or until the data ends). A client that returns later still counts as churned at that drop. | A one-month gap between one person leaving and a replacement starting would otherwise look like churn. Variant without confirmation in Robustness. |
| 9 | Horizon H: churned if the exit month is within 1–H months after the anchor. Only anchors with H months of data after them are used, so each horizon uses a different (earlier) set of anchors. | Avoids counting clients as "stayed" when we can't see far enough. |
| 10 | Anchors from **Jan 2018**. | Engagement volumes before 2018 are small. A 2021+ variant is in Robustness. |
| 12 | **Size groups overlap** (under 5 = 1–4 seats; 5+, 10+, 15+, 20+ are cumulative) and use the seat count held at the anchor. | "At least this many seats" is how the question was framed. |
| 13 | Profile features use the engagement file only, measured at the anchor month-end: years since the client's first seat, seats vs 6 months earlier, share below the client's peak so far, staff exits and starts in the last 12 months ÷ average seats, average months current staff have been in their seats, and previous confirmed exits. | Nothing after the anchor is used, so a profile could be applied to clients today. |
| 14 | A group's profile is read only with 20+ leavers. The client list (each client's latest spell) is written to `Desktop/Projects/Churn Analysis/Outputs/`, never into the notebook or the repo. | Small groups give unstable comparisons; client names stay out of git. |
| 11 | Dividing line = the "seats ≤ k" rule with the best Youden's J. AUC ranges come from resampling clients (400 draws; 200 in Robustness). | Clients with several spells would otherwise make the results look more certain than they are. |
| 15 | **Role source** = Staff Central *Staff Report with Details* (16 Nov 2026), cleaned and backfilled with `shared/jp_titles.py` (`backfill_jp`, default strict settings: ambiguous designations stay blank). Roles are compared at **title base** (seniority removed) and **JP category** level. | The engagement list has no titles. The strict backfill avoids guessed roles. |
| 16 | The staff report holds **one row per person** (their latest engagement). A seat's role is taken by **Engagement ID** when it matches, otherwise by **CSID** (the person's role today, which may be at another client). Every test holds this match type constant. | Older engagements are not in the report. The match type is tied to churn (the person left or moved), so ignoring it would bias the comparison. |
| 17 | **Unknown** role (no JP title/category even after backfill) is kept as its own group but not read as a role effect. | Unknown is far more common among churned-out seats, mostly because the record or person is gone: it follows from the outcome. |
| 18 | Part 3A uses 1-seat stable spells (Part 1 rules) with a full 12-month horizon. Categories with under 15 spells are grouped as Other; titles need 10+ spells. Significance is a likelihood-ratio test of a logistic regression with vs without category, holding anchor year and match type constant. | Small groups give unstable rates; one combined test avoids reading chance differences between many categories. |
| 19 | **Start-at-1 cohort** = first seat in Jan 2016 or later, and exactly 1 seat at each of the first 6 month-ends (month 1 = month of the first seat). Growth = **peak** seat count reached by month 12/24/36/48, each using only clients with that much data. The role of the first seat is the engagement active at month 6. | Matches the question ("start at 1 for about 6 months"). Using only clients old enough for each milestone avoids counting young clients as "didn't grow". |